# Paired volcano: univariate Cox at landmark 0d vs +90d

Time-to-platinum endpoint. Each dot is one (lab × feature_stat) pair.
Color encodes clinical category (CBC, CMP, LFT, Vitals, Androgen axis, Other).
ns features (q ≥ 0.05) are faded gray; q-significant features are bold in
their category color.

Three narrative beats highlighted on the figure:
- **Testosterone** is a key indicator at both landmarks
- **PSA** becomes significant at +90d
- **CBC + LFT** signals (general health markers) emerge by +90d


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})


## Configure paths

Edit `UNI_PATH` to point at your `cox_agg_univariate_nobs_adjusted.csv`.


In [ ]:
UNI_PATH_0  = Path(
    "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/"
    "abstract_final_survival_analysis/local_runs/cox/landmark_0/both/"
    "cox_agg_univariate_nobs_adjusted.csv"
)
UNI_PATH_90 = Path(
    "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/"
    "abstract_final_survival_analysis/local_runs/cox/landmark_90/both/"
    "cox_agg_univariate_nobs_adjusted.csv"
)
OUT_DIR = Path("./figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)


## Clinical category mapping

40 labs in six buckets. `Body height` is dropped due to data-quality concerns.
PT is folded into LFT since it tracks hepatic synthetic function.


In [ ]:
CBC = {
    "WBC", "RBC", "Hemoglobin", "Hematocrit",
    "MCV", "MCH", "MCHC", "RDW", "Platelets",
    "Neutrophils absolute", "Lymphocytes absolute",
    "Monocytes absolute", "Eosinophils absolute", "Basophils absolute",
}
CMP = {"Sodium", "Potassium", "Chloride", "CO2",
       "BUN", "Creatinine", "Glucose", "Calcium"}
LFT = {"ALT", "AST", "Alkaline phosphatase",
       "Total bilirubin", "Direct bilirubin",
       "Albumin", "Globulin", "Total protein", "PT"}
VITALS = {"Body weight", "Body temperature", "Heart rate",
          "Respiratory rate", "Systolic blood pressure",
          "Diastolic blood pressure"}
ANDROGEN = {"PSA", "Testosterone"}
OTHER = {"TSH"}
DROP = {"Body height"}

CATEGORY_MAP: dict[str, str] = {}
for label, members in [
    ("CBC", CBC), ("CMP", CMP), ("LFT", LFT),
    ("Vitals", VITALS), ("Androgen axis", ANDROGEN), ("Other", OTHER),
]:
    for m in members:
        CATEGORY_MAP[m] = label

DRAW_ORDER = ["Other", "Vitals", "CMP", "LFT", "CBC", "Androgen axis"]
LEGEND_ORDER = ["Androgen axis", "CBC", "LFT", "CMP", "Vitals", "Other"]

CATEGORY_COLORS = {
    "Androgen axis": "#8e1c2b",
    "CBC":           "#16a085",
    "LFT":           "#e67e22",
    "CMP":           "#7d3c98",
    "Vitals":        "#5d6d7e",
    "Other":         "#95a5a6",
}
NS_COLOR = "#d5d8dc"

def assign_category(lab_name: str) -> str:
    return CATEGORY_MAP.get(lab_name, "Other")


## Plotting helpers

If you have `adjustText` installed (`pip install adjustText`), labels will
be auto-placed without overlap. Otherwise the script falls back to a
deterministic offset rotation — readable, but may collide in dense regions.


In [ ]:
# ----------------------------- labeling knobs ---------------------------
TOP_K_PER_PANEL = 6
ALWAYS_LABEL = {"Testosterone", "PSA", "Hemoglobin", "Albumin",
                "Alkaline phosphatase"}
LABEL_FORMAT = "{lab} ({stat})"
LABEL_FONTSIZE = 8.5


def q_threshold_neglog10p(sub):
    """y-value (-log10 p) at which q just crosses 0.05; None if no q-sig features."""
    sig = sub.loc[sub["q_value"] < 0.05, "p_value"]
    if sig.empty:
        return None
    cutoff_p = float(sig.max())
    return float(-np.log10(max(cutoff_p, 1e-300)))


def _auto_label(ax, sub, *, top_k, always_label, label_format, curated_labs):
    """Label up to top_k labs (deduped to most-sig stat per lab), plus any
    significant labs in the always_label whitelist. Skips labs already
    labeled by curated narrative annotations (curated_labs)."""
    sig = sub.loc[sub["sig"] & ~sub["lab_name"].isin(curated_labs)].copy()
    if sig.empty:
        return

    best_per_lab = (sig.sort_values("p_value")
                       .drop_duplicates("lab_name", keep="first"))
    always_sig = best_per_lab.loc[best_per_lab["lab_name"].isin(always_label)]
    extra = (best_per_lab
             .loc[~best_per_lab["lab_name"].isin(always_label)]
             .head(top_k))
    to_label = pd.concat([always_sig, extra]).drop_duplicates("lab_name")
    if to_label.empty:
        return

    xs = to_label["hazard_ratio_per_sd"].to_numpy()
    ys = to_label["neglog10p"].to_numpy()
    label_strs = [label_format.format(lab=r["lab_name"],
                                      stat=r["feature_stat"])
                  for _, r in to_label.iterrows()]
    colors = [CATEGORY_COLORS[c] for c in to_label["category"]]

    try:
        from adjustText import adjust_text
        texts = [ax.text(x, y, s, fontsize=LABEL_FONTSIZE, color=c,
                         weight="semibold", zorder=10)
                 for x, y, s, c in zip(xs, ys, label_strs, colors)]
        adjust_text(
            texts, ax=ax,
            arrowprops=dict(arrowstyle="-", color="#7f8c8d", lw=0.5),
            expand_points=(1.4, 1.6), expand_text=(1.2, 1.4),
            only_move={"points": "y", "text": "xy"},
        )
    except ImportError:
        # fallback: deterministic offset rotation, no collision avoidance
        for i, (x, y, s, c) in enumerate(zip(xs, ys, label_strs, colors)):
            angle = np.deg2rad((i * 137) % 360)
            dx, dy = 16 * np.cos(angle), 16 * np.sin(angle)
            ax.annotate(
                s, (x, y),
                xytext=(dx, dy), textcoords="offset points",
                fontsize=LABEL_FONTSIZE, color=c, weight="semibold",
                zorder=10,
                arrowprops=dict(arrowstyle="-", lw=0.5, color="#95a5a6"),
            )


def plot_panel(ax, sub, title, *, label_testosterone, label_psa,
               bracket_general_health):
    sub = sub.loc[~sub["lab_name"].isin(DROP)].copy()
    sub["category"] = sub["lab_name"].map(assign_category)
    sub["neglog10p"] = -np.log10(sub["p_value"].clip(lower=1e-300))
    sub["sig"] = sub["q_value"] < 0.05

    ns = sub.loc[~sub["sig"]]
    ax.scatter(ns["hazard_ratio_per_sd"], ns["neglog10p"],
               s=20, color=NS_COLOR, alpha=0.45,
               edgecolor="none", zorder=1)

    for cat in DRAW_ORDER:
        cat_sig = sub.loc[sub["sig"] & (sub["category"] == cat)]
        if cat_sig.empty:
            continue
        is_hero = cat == "Androgen axis"
        ax.scatter(
            cat_sig["hazard_ratio_per_sd"], cat_sig["neglog10p"],
            s=80 if is_hero else 40,
            color=CATEGORY_COLORS[cat],
            alpha=0.92,
            edgecolor="white", linewidth=0.6,
            zorder=5 if is_hero else 3,
        )

    ax.axvline(1, color="grey", linestyle="-", linewidth=0.6, zorder=0)
    ax.set_xscale("log")
    q_y = q_threshold_neglog10p(sub)
    if q_y is not None:
        ax.axhline(q_y, color="black", linestyle=":", linewidth=0.9, zorder=0)

    hero_color = CATEGORY_COLORS["Androgen axis"]
    curated_labs = set()

    if label_testosterone:
        t = sub.loc[(sub["lab_name"] == "Testosterone") & sub["sig"]]
        if not t.empty:
            r = t.loc[t["neglog10p"].idxmax()]
            ax.annotate(
                f"Testosterone {r['feature_stat']}",
                (r["hazard_ratio_per_sd"], r["neglog10p"]),
                xytext=(14, 12), textcoords="offset points",
                fontsize=10.5, weight="bold", color=hero_color,
                arrowprops=dict(arrowstyle="-", lw=0.9, color=hero_color),
            )
            curated_labs.add("Testosterone")

    if label_psa:
        p = sub.loc[(sub["lab_name"] == "PSA") & sub["sig"]]
        if not p.empty:
            r = p.loc[p["neglog10p"].idxmax()]
            ax.annotate(
                f"PSA {r['feature_stat']}",
                (r["hazard_ratio_per_sd"], r["neglog10p"]),
                xytext=(14, -22), textcoords="offset points",
                fontsize=10.5, weight="bold", color=hero_color,
                arrowprops=dict(arrowstyle="-", lw=0.9, color=hero_color),
            )
            curated_labs.add("PSA")

    if bracket_general_health:
        cluster = sub.loc[sub["sig"] & sub["category"].isin(["CBC", "LFT"])]
        if not cluster.empty:
            cx = float(cluster["hazard_ratio_per_sd"].median())
            cy = float(cluster["neglog10p"].median())
            ax.annotate(
                "CBC + LFT signals\nemerge by 90d",
                xy=(cx, cy), xycoords="data",
                xytext=(cx * 1.6, cy + 5),
                textcoords="data",
                ha="left", va="bottom",
                fontsize=10, style="italic", color="#2c3e50",
                arrowprops=dict(arrowstyle="-|>", lw=1.0, color="#2c3e50",
                                connectionstyle="arc3,rad=0.18"),
            )

    # auto-label additional top hits (deduped by lab) + always-label whitelist
    _auto_label(ax, sub,
                top_k=TOP_K_PER_PANEL,
                always_label=ALWAYS_LABEL,
                label_format=LABEL_FORMAT,
                curated_labs=curated_labs)

    ax.set_xlabel("HR per SD (log scale)")
    ax.set_ylabel(r"$-\log_{10}(p)$")
    ax.set_title(title, fontsize=12.5, weight="bold")

    n_tested = len(sub)
    n_sig = int(sub["sig"].sum())
    breakdown = (sub.loc[sub["sig"], "category"]
                 .value_counts()
                 .reindex([c for c in LEGEND_ORDER if c != "Other"], fill_value=0))
    short = {"Androgen axis": "Androgen", "CBC": "CBC", "LFT": "LFT",
             "CMP": "CMP", "Vitals": "Vitals"}
    bd_str = "  ".join(f"{short[c]} {int(n)}" for c, n in breakdown.items())
    ax.text(0.99, 0.02,
            f"{n_sig} / {n_tested} q<0.05   ·   {bd_str}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=8.5, color="#5d6d7e", family="monospace")


## Load and filter data

Read the per-landmark files, tag each with its `landmark_days` (authoritative
from the file path), concatenate, keep `endpoint == "platinum"`.


In [ ]:
def _load(path, landmark_days):
    df = pd.read_csv(path)
    df["landmark_days"] = landmark_days
    return df

uni = pd.concat(
    [_load(UNI_PATH_0, 0), _load(UNI_PATH_90, 90)],
    ignore_index=True,
)
uni = uni.loc[uni["endpoint"] == "platinum"].copy()
uni = uni.dropna(subset=["coef_feature", "p_value", "q_value"])
print(f"{len(uni)} (lab × stat) rows across landmarks "
      f"{sorted(uni['landmark_days'].unique())}")
uni.head()


## Filter unstable Cox estimates

Drop rows where the Cox MLE blew up (|log HR| > 2 or 95% CI spans >2 orders
of magnitude). These are sparsely-observed features the model couldn't
actually estimate; they stretch the x-axis and obscure the real signal.


In [ ]:
COEF_CAP = 2.0     # |log HR per SD| above this is almost always pathological
CI_RATIO_CAP = 100  # CI upper / CI lower above this = essentially unestimated

ci_ratio = uni["ci_upper"] / uni["ci_lower"]
mask = (uni["coef_feature"].abs() <= COEF_CAP) & (ci_ratio < CI_RATIO_CAP)

dropped = uni.loc[~mask, [
    "lab_name", "feature_stat", "landmark_days",
    "coef_feature", "ci_lower", "ci_upper", "n_patients_used",
]].sort_values("coef_feature", key=abs, ascending=False)

print(f"dropping {len(dropped)} / {len(uni)} unstable rows")
display(dropped.head(15))

uni = uni.loc[mask].copy()
print(f"{len(uni)} rows remaining")


## Render the paired volcano


In [ ]:
panels = [
    (0,  "Landmark = 0 days",
     dict(label_testosterone=True, label_psa=False,
          bracket_general_health=False)),
    (90, "Landmark = +90 days",
     dict(label_testosterone=True, label_psa=True,
          bracket_general_health=True)),
]

fig, axes = plt.subplots(
    1, 2, figsize=(14, 6),
    sharex=True, sharey=True,
    gridspec_kw={"wspace": 0.08},
)

for ax, (lm, title, kwargs) in zip(axes, panels):
    sub = uni.loc[uni["landmark_days"] == lm]
    if sub.empty:
        ax.text(0.5, 0.5, f"(no data for landmark = {lm}d)",
                ha="center", va="center", transform=ax.transAxes,
                color="#7f8c8d")
        ax.set_axis_off()
        continue
    plot_panel(ax, sub, title, **kwargs)

for ax in axes:
    ax.set_xlim(0.22, 4.5)  # HR per SD; log-scaled axis

fig.suptitle(
    "Univariate Cox (n_obs-adjusted) — time-to-platinum",
    fontsize=15, weight="bold", y=1.02,
)

handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white",
                 label=c) for c in LEGEND_ORDER]
handles.append(Line2D([0], [0], marker="o", color="w",
                      markerfacecolor=NS_COLOR, markersize=8,
                      label="ns (q ≥ 0.05)"))
fig.legend(handles=handles, loc="lower center",
           ncol=len(handles), bbox_to_anchor=(0.5, -0.04),
           fontsize=10)

plt.show()


## Save figure


In [ ]:
for ext in ("png", "pdf"):
    out = OUT_DIR / f"paired_volcano_platinum.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
